# Где ломалась обработка ЭКГ и что изменилось

Шесть пошаговых примеров на **реальных записях** из локального датасета. Каждый пример содержит причину, графики «до/после», численную проверку и результат. QC означает проверку качества окна. Его решение «принять/отклонить» не является меткой приступа.

Последовательность обработки: **правильное чтение → QC исходных отсчётов → ресэмплинг → фильтр → QC результата → нормализация**. В примерах графики остаются в мкВ, чтобы нормализация не скрывала дефекты.

**Запуск в VS Code:** выберите ядро `.venv`, перезапустите его и выполните Run All. Нужны локальные `data/catalog.sqlite`, исходные `.e` и зависимости `pip install -e ".[notebook]"`. Модели не обучаются; каталог открывается только для чтения. Номера записей ниже относятся к текущему каталогу: для другого датасета интервалы нужно выбрать заново.

Старый код читается прямо из Git-коммита `2c7e1c0b343c6132749acb4d41281f212dfe6df6`; исправленный ридер — из `epi`, а фильтрация/QC зафиксированы на коммите `2e4e0ea`. Старый ридер использует установленный `pynicolet` (при проверке: 0.1.2); версия выводится ниже. Никакие сигналы не генерируются и не подменяются. Для сравнения чтения используется Neo.

В Git хранится этот ноутбук без выводов. Выполненная копия с графиками сохраняется локально в `data/reports/ecg_failure_examples/`.

**Это историческое сравнение до добавления интерполяции.** Его шесть примеров
фиксируют обработку из `2e4e0ea`, которая отклоняла окно с коротким выбросом.
Новый режим по умолчанию и маски замен показаны отдельно:
[замена выбросов до фильтрации](epi_ecg_spike_repair.ipynb).


In [ ]:
from pathlib import Path
from fractions import Fraction
from functools import lru_cache
from importlib.metadata import version
import sqlite3
import subprocess
import sys
import types
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import resample_poly, welch
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "epi/__init__.py").exists())
sys.path.insert(0, str(ROOT))
from epi.reader import NicoletEReader
from epi.preprocessing import (
    preprocess_ecg, ecg_quality_report, ecg_raw_quality_report,
    line_noise_power_ratio,
)
from neo.rawio import NicoletRawIO

CATALOG = ROOT / "data/catalog.sqlite"
OUT = ROOT / "data/reports/ecg_failure_examples"
OUT.mkdir(parents=True, exist_ok=True)
assert CATALOG.is_file(), "Сначала создайте локальный data/catalog.sqlite"
BASELINE = "2c7e1c0b343c6132749acb4d41281f212dfe6df6"
TARGET_RATE = 256
results = []

def baseline_module(name, revision=BASELINE):
    ref = f"{revision}:epi/{name}.py"
    result = subprocess.run(["git", "show", ref], cwd=ROOT, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Нет исходной версии в локальной истории Git. Нужен коммит " + BASELINE)
    module = types.ModuleType("baseline_" + name)
    exec(compile(result.stdout, ref, "exec"), module.__dict__)
    return module

old_reader_module = baseline_module("reader")
old_preprocessing = baseline_module("preprocessing")
print("Старый код:", BASELINE[:8])
print("pynicolet:", version("pynicolet"), "neo:", version("neo"))

filter_only = baseline_module("preprocessing", "2e4e0eafa01bc1a9ee850d425f27c0ad4c5ea447")
preprocess_ecg = filter_only.preprocess_ecg
ecg_quality_report = filter_only.ecg_quality_report
ecg_raw_quality_report = filter_only.ecg_raw_quality_report


## Общие функции

Эта клетка только определяет чтение и оформление графиков. `read_case` читает один интервал, подаёт **одинаковые отсчёты** в старый и новый фильтр и сохраняет обе проверки QC. Пределы АЦП берутся из заголовка записи и проверяются до ресэмплинга.

`old_native` отдельно воспроизводит старую ошибку чтения; оно не используется как вход для сравнения фильтров. Так исправление чтения не смешивается с изменением фильтрации.

In [ ]:
@lru_cache(maxsize=16)
def readers(recording_id):
    with sqlite3.connect(f"file:{CATALOG}?mode=ro", uri=True) as conn:
        root = Path(conn.execute("SELECT value FROM meta WHERE key='root'").fetchone()[0])
        row = conn.execute("SELECT rel_path FROM recordings WHERE id=? AND deleted_at IS NULL",
                           (int(recording_id),)).fetchone()
    if row is None:
        raise ValueError(f"Записи {recording_id} нет в текущем каталоге")
    path = root / row[0]
    return (NicoletEReader(path, type_names={}),
            old_reader_module.NicoletEReader(path, type_names={}))

def to_target(native, rate):
    ratio = Fraction(TARGET_RATE / rate).limit_denominator(10000)
    return resample_poly(native, ratio.numerator, ratio.denominator, axis=-1).astype("float32")

def reason_text(report):
    value = report["reasons"]
    return ";".join(value) or "ok" if isinstance(value, list) else value

def read_case(recording_id, segment, start, duration):
    reader, legacy = readers(recording_id)
    reader.select_segment(segment)
    legacy.select_segment(segment)
    native = reader.read_channel("EKG", start, duration)[None]
    raw = to_target(native, reader.sampling_rate)
    before = old_preprocessing.preprocess(raw, TARGET_RATE)
    after = preprocess_ecg(raw, TARGET_RATE)
    source = ecg_raw_quality_report(native, reader.sampling_rate,
                                    adc_limits=reader.channel_limits("EKG"))
    output = ecg_quality_report(raw, TARGET_RATE, filtered=after)
    reasons = sorted(set((reason_text(source) + ";" + reason_text(output)).split(";")) - {"ok"})
    return dict(recording_id=recording_id, segment=segment, start=start, duration=duration,
                reader=reader, legacy=legacy, rate=reader.sampling_rate,
                native=native, raw=raw, before=before, after=after,
                source_qc=source, output_qc=output,
                old_raw_qc=old_preprocessing.quality_report(raw),
                old_clean_qc=old_preprocessing.quality_report(before),
                keep=bool(source["keep"] and output["keep"]), reasons=";".join(reasons) or "ok")

def old_native(case):
    case["legacy"].select_segment(case["segment"])
    return case["legacy"].read_channel("EKG", case["start"], case["duration"])[None]

def neo_reference(case):
    with warnings.catch_warnings(record=True) as caught:
        backend = NicoletRawIO(filename=str(case["reader"].path))
        backend.parse_header()
        backend.set_signal_channels(case["segment"])
        channels = backend.header["signal_channels"]
        channel = channels[channels["name"] == "EKG"][0]
        stream = int(np.flatnonzero(backend.header["signal_streams"]["id"] == channel["stream_id"])[0])
        stream_channels = channels[channels["stream_id"] == channel["stream_id"]]
        index = int(np.flatnonzero(stream_channels["name"] == "EKG")[0])
        first = int(round(case["start"] * case["rate"]))
        raw = backend.get_analogsignal_chunk(0, case["segment"], first,
            first + case["native"].shape[-1], stream_index=stream, channel_indexes=[index])
        reference = backend.rescale_signal_raw_to_float(raw, dtype="float64",
            stream_index=stream, channel_indexes=[index])[:, 0]
        reference *= {"V": 1e6, "mV": 1000}.get(str(channel["units"]), 1)
    print("Предупреждений Neo при чтении заголовка:", len(caught))
    return reference

def plot_processing(case, filename):
    t = case["start"] + np.arange(case["raw"].shape[-1]) / TARGET_RATE
    fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True, layout="constrained")
    titles = ["Одинаковый вход для обоих фильтров, 256 Гц",
              "Старый фильтр — решение старого QC: " + str(case["old_raw_qc"]["keep"]),
              "Новый фильтр — итоговый QC: " + ("ПРИНЯТЬ" if case["keep"] else "ОТКЛОНИТЬ")]
    for ax, signal, title in zip(axes, [case["raw"], case["before"], case["after"]], titles):
        ax.plot(t, signal[0], lw=.75)
        ax.set(title=title, ylabel="мкВ")
        ax.grid(alpha=.2)
    limits = (min(axes[1].get_ylim()[0], axes[2].get_ylim()[0]),
              max(axes[1].get_ylim()[1], axes[2].get_ylim()[1]))
    axes[1].set_ylim(limits)
    axes[2].set_ylim(limits)
    axes[-1].set_facecolor("#edf6ee" if case["keep"] else "#fff0ef")
    axes[-1].set_xlabel("Секунды от начала сегмента")
    fig.savefig(OUT / filename, dpi=120)
    plt.show()

def show_qc(case):
    rows = []
    for stage, report in [("Старый QC: до фильтра", case["old_raw_qc"]),
                          ("Старый QC: после фильтра (для сравнения)", case["old_clean_qc"]),
                          ("Новый QC: исходные отсчёты", case["source_qc"]),
                          ("Новый QC: результат фильтра", case["output_qc"])]:
        rows.append({"этап": stage, "принять": report["keep"], "причины": reason_text(report),
                     "доля нулей": report["flat_fraction"],
                     "амплитуда p99.5, мкВ": report["amplitude_uv"]})
    display(pd.DataFrame(rows))
    print("Итог:", "ПРИНЯТЬ" if case["keep"] else "ОТКЛОНИТЬ", "—", case["reasons"])

## 1. Вместо живой ЭКГ ридер возвращал нули

**Запись 44, сегмент 1, 100–110 с.** Старый ридер неверно учитывал смещение позднего сегмента в последовательности пакетов. Запрошенный фрагмент существует, но библиотека возвращала нули.

**Исправление:** смещение сегмента учитывается ровно один раз, включая позицию внутри пакета. Сравниваем старое чтение, исправленное чтение и независимый ридер Neo. Это пример исправления чтения; фильтр здесь не применяется.

In [ ]:
reading = read_case(44, 1, 100.0, 10.0)
legacy_signal = old_native(reading)[0]
reference = neo_reference(reading)
fixed_signal = reading["native"][0]
reader_table = pd.DataFrame([
    {"чтение": name, "отсчётов": len(signal), "нулей, %": 100 * np.mean(signal == 0),
     "макс. отличие от Neo, мкВ": np.max(np.abs(signal - reference))}
    for name, signal in [("Старое", legacy_signal), ("Исправленное", fixed_signal), ("Neo", reference)]
])
display(reader_table)
t = reading["start"] + np.arange(len(reference)) / reading["rate"]
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True, layout="constrained")
axes[0].plot(t, reference, color="#248269", lw=1.5, label="Neo")
axes[0].plot(t, fixed_signal, color="#755ca1", lw=.8, ls="--", label="Исправленное чтение")
axes[0].plot(t, legacy_signal, color="#bb433c", lw=1.3, label="Старое чтение")
axes[1].plot(t, legacy_signal-reference, color="#bb433c", lw=.7, label="Ошибка старого чтения")
axes[1].plot(t, fixed_signal-reference, color="#755ca1", lw=1, label="Ошибка исправленного чтения")
for ax in axes:
    ax.set_ylabel("мкВ")
    ax.grid(alpha=.2)
    ax.legend()
axes[-1].set_xlabel("Секунды от начала сегмента")
fig.savefig(OUT / "01_reader.png", dpi=120)
plt.show()
assert np.mean(legacy_signal == 0) == 1.0, "Старый backend изменился: проверьте версию pynicolet"
assert np.max(np.abs(fixed_signal-reference)) < 1e-6
print("Исправленное чтение совпадает с Neo; живой сигнал восстановлен из исходного файла.")
results.append({"пример": "Чтение позднего сегмента", "было": "100% нулей", "стало": "Совпадает с Neo"})

## 2. Большой одиночный выброс проходил в обучение

**Запись 85, сегмент 0, 405–415 с.** Старый QC использовал 99.5-й процентиль: короткий импульс мог слабо влиять на эту величину. Кроме того, предупреждение о скачке не меняло `keep` на False. После фильтра выброс оставался.

**Исправление:** сравниваем максимальный пик с типичной амплитудой секундных пиков. Окно отклоняется, когда одновременно пик больше **1000 мкВ** и больше типичного пика в **8 раз**. Это правило отбраковки, а не удаление импульса из сигнала.

In [ ]:
spike = read_case(85, 0, 405.0, 10.0)
plot_processing(spike, "02_spike.png")
show_qc(spike)
display(pd.DataFrame([{
    "пик, мкВ": spike["output_qc"]["peak_uv"],
    "пик / типичный секундный пик": spike["output_qc"]["peak_ratio"],
    "порог отношения": 8.0,
}]))
assert spike["old_raw_qc"]["keep"] and spike["old_clean_qc"]["keep"]
assert not spike["keep"] and "isolated_spike" in spike["reasons"]
results.append({"пример": "Одиночный выброс", "было": "Окно принималось", "стало": "Отклонено: isolated_spike"})

## 3. Сигнал достигал предела АЦП, но окно оставалось допустимым

**Запись 210, сегмент 0, 4475–4485 с.** АЦП — устройство, переводящее напряжение в числа. При достижении его предела верхушки сигнала обрезаются. Фильтрация не может узнать, какой была потерянная часть. Ресэмплинг сглаживает плато, поэтому проверять насыщение нужно по исходным отсчётам.

**Исправление:** читаем физические пределы АЦП из заголовка и отклоняем окно, если на них находится больше 1% отсчётов. Старый QC оставлял клиппинг предупреждением.

In [ ]:
saturation = read_case(210, 0, 4475.0, 10.0)
lower, upper = saturation["reader"].channel_limits("EKG")
x = saturation["native"][0]
t = saturation["start"] + np.arange(len(x)) / saturation["rate"]
rail = (x <= lower + 1e-6) | (x >= upper - 1e-6)
center = t[np.flatnonzero(rail)[0]]
zoom = (t >= center - .05) & (t <= center + .15)
fig, axes = plt.subplots(2, 1, figsize=(13, 6), layout="constrained")
for ax, mask, title in [(axes[0], np.ones(len(x), dtype=bool), "Исходная запись: красные точки на пределах АЦП"),
                        (axes[1], zoom, "Увеличенный участок с обрезанными верхушками")]:
    ax.plot(t[mask], x[mask], lw=.7)
    ax.scatter(t[mask & rail], x[mask & rail], color="#bb433c", s=9, zorder=3)
    for limit in [lower, upper]:
        ax.axhline(limit, color="#bb433c", ls="--", alpha=.7)
    ax.set(title=title, ylabel="мкВ", xlabel="Секунды от начала сегмента")
    ax.grid(alpha=.2)
fig.savefig(OUT / "03_adc_clipping.png", dpi=120)
plt.show()
plot_processing(saturation, "03_filtered_clipping.png")
show_qc(saturation)
print(f"На пределах АЦП: {100 * saturation['source_qc']['adc_rail_fraction']:.2f}% отсчётов; порог: 1%")
assert saturation["old_raw_qc"]["keep"]
assert not saturation["keep"] and "adc_clipping" in saturation["reasons"]
results.append({"пример": "Насыщение АЦП", "было": "Предупреждение, окно принималось", "стало": "Отклонено по исходным отсчётам"})

## 4. Почему нельзя проверять пропуски только после фильтра

**Запись 46, сегмент 5, 4745–4755 с.** Старый ридер возвращал здесь фрагмент, примерно на 90% состоящий из нулей. Фильтр размазывал переход к нулям: точные нули исчезали, и повторный QC только по результату фильтра мог пропустить окно.

В исходном модельном ноутбуке QC уже вызывался по исходному массиву. Поэтому этот пример показывает опасность перестановки этапов, а не дополнительную ошибку именно того ноутбука.

**Защита:** решение по исходным отсчётам сохраняется и объединяется с QC результата. Отдельно исправлено чтение самого сегмента, поэтому искусственные нули больше не возникают в этом месте.

In [ ]:
dropout = read_case(46, 5, 4745.0, 10.0)
damaged = to_target(old_native(dropout), dropout["rate"])
filtered_damaged = old_preprocessing.preprocess(damaged, TARGET_RATE)
before_raw = old_preprocessing.quality_report(damaged)
after_only = old_preprocessing.quality_report(filtered_damaged)
protected = ecg_quality_report(damaged, TARGET_RATE, filtered=preprocess_ecg(damaged, TARGET_RATE))
display(pd.DataFrame([
    {"проверка": name, "доля нулей": report["flat_fraction"],
     "принять": report["keep"], "причины": reason_text(report)}
    for name, report in [("Старый QC до фильтра", before_raw),
                         ("Старый QC только после фильтра", after_only),
                         ("Новый QC с сохранением исходного сигнала", protected)]
]))
t = dropout["start"] + np.arange(damaged.shape[-1]) / TARGET_RATE
fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True, layout="constrained")
for ax, signal, title in zip(axes, [damaged, filtered_damaged, dropout["raw"]],
    ["Старое чтение: большая часть окна — нули", "После старого фильтра: точные нули исчезли",
     "Исправленное чтение того же интервала, до фильтра"]):
    ax.plot(t, signal[0], lw=.8)
    ax.set(title=title, ylabel="мкВ")
    ax.grid(alpha=.2)
axes[-1].set_xlabel("Секунды от начала сегмента")
fig.savefig(OUT / "04_raw_qc.png", dpi=120)
plt.show()
assert not before_raw["keep"] and after_only["keep"]
assert not protected["keep"]
assert np.mean(dropout["raw"] == 0) < np.mean(damaged == 0)
print("Даже если фильтр скрыл точные нули, исходное плохое окно остаётся отклонённым.")
results.append({"пример": "QC только после фильтра", "было": "Пропуск скрывался фильтром", "стало": "Решение исходного QC сохраняется"})

## 5. Сетевая помеха: изменение фильтра видно в спектре

**Запись 85, сегмент 0, 740–1040 с.** Это полное 300-секундное окно. Старый фильтр использовал узкий notch 50 Гц, новый подавляет полосу 48–52 Гц. Затем в обоих случаях применяется полоса 0.5–40 Гц; в новом варианте добавлено отражение краёв.

Оба фильтра получают одно окно. Метрика одна и та же: **P(48–52 Гц) / P(0.5–40 Гц)**, Welch, отрезки 4 с; края по 2 с исключаются только из оценки спектра. Старая неизвестная функция `line_noise_ratio` из epi 1.3.0 здесь не используется. Снижение показателя само по себе не доказывает наличие качественной ЭКГ или улучшение модели.

In [ ]:
noise = read_case(85, 0, 740.0, 300.0)
noise_values = {name: line_noise_power_ratio(x, TARGET_RATE)
                for name, x in [("До фильтра", noise["raw"]),
                                ("Старый фильтр", noise["before"]),
                                ("Новый фильтр", noise["after"])]}
display(pd.DataFrame({"этап": list(noise_values), "отношение мощностей": list(noise_values.values())}))
reduction = noise_values["Старый фильтр"] / noise_values["Новый фильтр"]
print(f"Показатель остаточной сетевой помехи уменьшился в {reduction:.2f} раза")
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), layout="constrained")
for name, x in [("До фильтра", noise["raw"]), ("Старый фильтр", noise["before"]), ("Новый фильтр", noise["after"])]:
    freq, psd = welch(x[0, 2*TARGET_RATE:-2*TARGET_RATE], fs=TARGET_RATE, nperseg=4*TARGET_RATE)
    for ax in axes:
        ax.semilogy(freq, np.maximum(psd, np.finfo(float).tiny), label=name)
for ax, limits in zip(axes, [(0.5, 70), (44, 56)]):
    ax.set_xlim(limits)
    ax.axvspan(48, 52, color="#f0c674", alpha=.25)
    ax.set(xlabel="Частота, Гц", ylabel="Спектральная плотность, мкВ²/Гц")
    ax.grid(alpha=.2)
    ax.legend()
fig.savefig(OUT / "05_mains_spectrum.png", dpi=120)
plt.show()
assert noise_values["Новый фильтр"] < noise_values["Старый фильтр"]
results.append({"пример": "Сетевая помеха", "было": "Узкий notch 50 Гц", "стало": f"Показатель ниже в {reduction:.2f} раза"})

## 6. Проверка, что обычные высокие R-зубцы сохраняются

**Запись 31, сегмент 1, 1665–1675 с.** На графике видны повторяющиеся сердечные комплексы. Большая амплитуда и резкие фронты могут быть свойством полезного сигнала. Простое включение `strict=True` для всех старых предупреждений исключило бы и этот пример.

Новая проверка одиночного выброса сравнивает пик с типичным секундным пиком этого же окна. Для повторяющихся комплексов отношение близко к единице. Это контрольный пример против чрезмерной отбраковки; он не заменяет экспертную оценку всех записей.

In [ ]:
heart = read_case(31, 1, 1665.0, 10.0)
plot_processing(heart, "06_heart_preserved.png")
show_qc(heart)
strict_keep = old_preprocessing.clean_mask(heart["raw"], strict=True)
display(pd.DataFrame([{
    "старый QC keep": heart["old_raw_qc"]["keep"],
    "старый QC strict=True": strict_keep,
    "новый итоговый QC": heart["keep"],
    "пик, мкВ": heart["output_qc"]["peak_uv"],
    "пик / типичный секундный пик": heart["output_qc"]["peak_ratio"],
}]))
assert not strict_keep
assert heart["keep"] and heart["output_qc"]["peak_ratio"] < 8
results.append({"пример": "Повторяющиеся R-зубцы", "было": "strict=True исключил бы окно", "стало": "Окно сохранено"})

## Что подтверждено этими примерами

Чтение позднего сегмента исправлено и совпадает с Neo. Выброс и насыщение теперь приводят к исключению окна. Исходный QC защищает от маскировки нулей фильтром. Полоса 48–52 Гц уменьшает остаточную сетевую мощность. Проверенный пример с повторяющимися сердечными комплексами сохранён.

Решение исключить окно не означает восстановление потерянных отсчётов. Пороговые правила требуют проверки на большем числе записей; здесь не измеряются чувствительность/специфичность QC и качество TCN. При обучении одна и та же маска применяется к сигналам, меткам, пациентам и метаданным, а обучаемые пороги определяются только на train.

In [ ]:
summary = pd.DataFrame(results)
display(summary)
assert len(summary) == 6
summary.to_csv(OUT / "examples_summary.csv", index=False)
print("Все шесть проверок выполнены. Графики и сводка сохранены локально в data/reports/ecg_failure_examples/.")